### The Lotka-Volterra Predator-Prey Model

Every other notebook in this topic integrates a single object moving under a
known force. This one models two populations that feed back on each other:
more prey means more predators, more predators means less prey, and less prey
means fewer predators. Neither equation can be solved on its own, and the result
is a cycle that repeats forever without anything driving it.

### The equations and what each term means

The Lotka-Volterra equations model the coupled population dynamics of
two interacting species - a prey population $x$ and a predator population $y$:

$$\frac{dx}{dt} = \alpha x - \beta x y
\qquad
\frac{dy}{dt} = \delta x y - \gamma y$$

Each term has a direct biological meaning:

- $\alpha x$ - prey reproduce at rate $\alpha$ when left alone
- $-\beta xy$ - prey are consumed by predators; rate proportional to encounters $xy$
- $\delta xy$ - predators gain population from eating prey
- $-\gamma y$ - predators die at rate $\gamma$ without food

The $xy$ products are the coupling. They assume predators and prey meet at a
rate proportional to how many of each are present, which is the same
mass-action reasoning used for reaction rates in chemistry.

The populations oscillate indefinitely with no friction or carrying capacity.

### Finding the equilibrium point

The **equilibrium point** (where both derivatives are zero) is found
by setting each equation to zero and solving.
From the prey equation:

$$0 = \alpha x - \beta x y = x(\alpha - \beta y)
\quad\Rightarrow\quad
y^* = \frac{\alpha}{\beta}$$

From the predator equation:

$$0 = \delta x y - \gamma y = y(\delta x - \gamma)
\quad\Rightarrow\quad
x^* = \frac{\gamma}{\delta}$$

(The trivial solution $x = y = 0$ - total extinction - is the other equilibrium.)
Both populations cycle around $(x^*, y^*)$ indefinitely.

Notice the crossover: the *prey* equilibrium $x^*$ depends only on the
*predator's* rate constants, and the predator equilibrium $y^*$ only on the
prey's. Feeding the prey better raises $\alpha$, which raises the average
predator population, not the average prey population.

### Why this needs a numerical solver

This is a **coupled nonlinear ODE system** - the equations cannot be solved
independently because $dx/dt$ depends on $y$ and $dy/dt$ depends on $x$.
We use `scipy.integrate.solve_ivp` with the **RKF45** method
(adaptive Runge-Kutta 4th/5th order) which automatically adjusts
its step size to maintain accuracy.

This is the first notebook in the topic to hand the stepping over to a library
rather than writing the loop by hand. Everything learned about step size and
accuracy still applies; RKF45 just chooses $\Delta t$ for us, step by step,
instead of holding it fixed.

---
### Setup: model parameters and initial conditions

All four rate constants are positive. Populations are expressed as fractions of
a reference level (1.0 = 100% of the reference population), so the results will
be plotted as percentages.

Using fractions rather than head counts matters here. The equations are
scale-free in the sense that they never refer to an absolute number of animals,
only to rates, so "100% of the reference population" is as meaningful a starting
point as any specific count would be.

The printout confirms the two equilibrium values computed from the rate
constants: $x^* = \gamma/\delta = 90\%$ for prey and
$y^* = \alpha/\beta = 182\%$ for predators. The simulation starts *away* from
that point (prey at 100%, predators at 50%), which is what sets the cycle going.
Start exactly at $(x^*, y^*)$ and both populations would sit there forever.

In [ ]:
"""predator_prey.ipynb"""

# Cell 01 - Model parameters and initial conditions

%matplotlib inline

import inspect

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import MultipleLocator
from scipy.integrate import solve_ivp

final_time = 20  # simulation duration (months)

alpha = 2.0  # prey birth rate (per month)
beta = 1.1  # prey death rate per predator-prey encounter
delta = 1.0  # predator birth rate per predator-prey encounter
gamma = 0.9  # predator death rate (per month)

# Equilibrium point: populations cycle around (x*, y*)
prey_eq = gamma / delta  # x* = gamma/delta
pred_eq = alpha / beta  # y* = alpha/beta

pred_0 = 0.5  # initial predator population (50% of reference)
prey_0 = 1.0  # initial prey population    (100% of reference)

print(f"\u03b1={alpha}, \u03b2={beta}, \u03b4={delta}, \u03b3={gamma}")
print(f"Initial predator : {pred_0:.2%}")
print(f"Initial prey     : {prey_0:.2%}")
print(f"Equilibrium prey : {prey_eq:.2%}  (x* = \u03b3/\u03b4)")
print(f"Equilibrium pred : {pred_eq:.2%}  (y* = \u03b1/\u03b2)")

---
### Defining the ODE system

`solve_ivp` expects a function `f(t, y)` that returns the derivatives
of the state vector.
The state vector here is `[pred, prey]`, so the function returns
`[d_pred/dt, d_prey/dt]` in the same order.
Extra constants ($\alpha, \beta, \delta, \gamma$) are passed through
the `args` keyword.

Keeping those two orderings consistent is the one place this cell can go wrong.
The solver has no idea which slot means what; it only guarantees that whatever
order you unpack the state in, it will write the returned derivatives back in
that same order. The cell prints the function's signature as a check that it
matches what `solve_ivp` will call.

Note also that `time` is accepted but never used. The Lotka-Volterra system is
**autonomous** - the rates depend on the populations alone, not on what the
clock reads - but `solve_ivp` still passes the time in, so the parameter has to
be there.

In [ ]:
# Cell 02 - Lotka-Volterra ODE system
# state_vector = [pred, prey]; returns [d_pred/dt, d_prey/dt]


def model(time, state_vector, alpha, beta, delta, gamma):
    pred, prey = state_vector
    d_prey = alpha * prey - beta * prey * pred  # dx/dt
    d_pred = delta * prey * pred - gamma * pred  # dy/dt
    return d_pred, d_prey


print(f"Function name  : {model.__name__}")
print(f"Signature      : {inspect.signature(model)}")

---
### Solving with `solve_ivp` (RKF45)

The Runge-Kutta-Fehlberg method (RKF45) uses a 4th-order solution to
advance the state and a 5th-order solution to estimate the local error.
If the error exceeds a tolerance the step is rejected and retried with
a smaller $\Delta t$.

This is the payoff of an adaptive method. The populations crawl along near their
minima and then spike sharply, so a fixed step size would either waste effort on
the flat stretches or under-resolve the peaks. RKF45 spends its steps where the
solution is actually changing.

`max_step` caps the step size to ensure enough resolution even in
slow-varying regions. Without it the solver would be free to take enormous
steps through the quiet parts of the cycle, and the resulting curve would be
accurate but too coarse to plot smoothly.

The solution arrives as `sol.t` and `sol.y`; multiplying by 100 converts the
fractional populations to the percentages used from here on.

In [ ]:
# Cell 03 - Solve the ODE system using RKF45

sol = solve_ivp(
    model,
    (0, final_time),  # time span
    [pred_0, prey_0],  # initial state vector [pred, prey]
    max_step=final_time / 1000,  # cap step size to 1/1000 of total time
    args=(alpha, beta, delta, gamma),
)

t = sol.t
pred, prey = sol.y * 100  # convert fractional populations to percent

pd.DataFrame({"time": t[:5], "predator %": pred[:5], "prey %": prey[:5]})

---
### Population dynamics over time

The predator and prey populations oscillate out of phase:
prey increase first (plentiful food, low predation),
which then drives predator growth (abundant prey),
which then suppresses the prey,
which then causes the predators to decline - and the cycle repeats.
The dashed horizontal lines mark the equilibrium values $x^*$ and $y^*$
around which both populations orbit.

The prey peak always arrives *before* the predator peak, never with it. That
lag is the whole mechanism: predators respond to how much food there was
recently, not to how much there is now, so they keep growing for a while after
the prey have already turned downward, and then overshoot on the way back.

With these rate constants the cycle takes about **5.4 months**, so the 20-month
run covers just under four full cycles. The swings are large: prey range from
about 11% to 317% of the reference population and predators from 50% to 450%.
Both curves come back to the same values every cycle rather than growing or
decaying, which is the signature of the model being conservative.

That last point connects this notebook to the rest of the topic. Just as the
undamped pendulum conserves energy, Lotka-Volterra conserves the quantity

$$V = \delta x - \gamma \ln x + \beta y - \alpha \ln y,$$

which is constant along any solution. It fixes the orbit the populations travel,
exactly as a fixed energy fixes the pendulum's phase-space loop. Over this run
RKF45 holds $V$ steady to about one part in $10^{10}$, so the cycle you see
repeat really is closing on itself and not slowly drifting.

In [ ]:
# Cell 04 - Plot predator and prey populations over time

plt.figure("predator_prey", figsize=(10, 5))
plt.plot(t, pred, label="Predator", color="red", linewidth=2)
plt.plot(t, prey, label="Prey", color="blue", linewidth=2)

# Mark the equilibrium levels
plt.axhline(
    prey_eq * 100,
    color="blue",
    linestyle=":",
    alpha=0.6,
    label=f"Prey equilibrium ({prey_eq:.0%})",
)
plt.axhline(
    pred_eq * 100,
    color="red",
    linestyle=":",
    alpha=0.6,
    label=f"Pred equilibrium ({pred_eq:.0%})",
)

plt.title("Predator-Prey Model (Lotka-Volterra)")
plt.xlabel("Time (months)")
plt.ylabel("Population (%)")

ax = plt.gca()
ax.xaxis.set_major_locator(MultipleLocator(5))
ax.xaxis.set_minor_locator(MultipleLocator(1))
ax.yaxis.set_major_locator(MultipleLocator(50))
ax.yaxis.set_minor_locator(MultipleLocator(10))
ax.legend(loc="upper right", framealpha=1.0, facecolor="white")

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

: 